# Week 6 · Topic 4 — Apply, Lambda and Finishing Up

**Box we are building: STORE + PROCESS, the whole job**

A `lambda` is a `def` you don't bother naming, for a one-line function you are
only going to use once — usually right inside `.apply()`.

> **Already know this?** Skip to the *Topic 4 Challenge*.

---
## 1. `.apply()` runs a function on every value in a column

Start with a normal function, the way Week 3 taught it.

In [1]:
import pandas as pd

servers = pd.read_csv("servers.csv")
servers["percent"] = (servers["used"] / servers["total"]) * 100

def classify(percent):
    if percent >= 100:
        return "OVER LIMIT"
    elif percent >= 90:
        return "WARNING"
    else:
        return "OK"

servers["status"] = servers["percent"].apply(classify)
print(servers)

  hostname region  used  total  percent      status
0   srv-01   east    87    100     87.0          OK
1   srv-02   east    45    100     45.0          OK
2   srv-03   west   120    100    120.0  OVER LIMIT
3   srv-04   west    93    100     93.0     WARNING
4   srv-05  north   101    100    101.0  OVER LIMIT
5   srv-06   east    60    100     60.0          OK
6   srv-07   west    30    100     30.0          OK
7   srv-08  north   110    100    110.0  OVER LIMIT
8   srv-09   east    95    100     95.0     WARNING
9   srv-10  north    55    100     55.0          OK


### Your turn
Write your own single-argument function and `.apply()` it to the `used` column.

In [2]:
def used_level(value):
    if value >= 100:
        return "at or over capacity"
    return "below capacity"

servers["used_level"] = servers["used"].apply(used_level)
print(servers[["hostname", "used", "used_level"]])

  hostname  used           used_level
0   srv-01    87       below capacity
1   srv-02    45       below capacity
2   srv-03   120  at or over capacity
3   srv-04    93       below capacity
4   srv-05   101  at or over capacity
5   srv-06    60       below capacity
6   srv-07    30       below capacity
7   srv-08   110  at or over capacity
8   srv-09    95       below capacity
9   srv-10    55       below capacity


---
## 2. `lambda` - the same idea, written inline

`lambda parameters: expression` — that IS the whole function. Only one
expression allowed — no `if`/`elif`/`else` statements, no loops.

In [3]:
double = lambda x: x * 2
print(double(21))     # 42 - same as def double(x): return x * 2

42


```
def classify(p):                    lambda p: (
    if p >= 100:                        "OVER LIMIT" if p >= 100
        return "OVER LIMIT"             else "WARNING" if p >= 90
    elif p >= 90:                       else "OK"
        return "WARNING"             )
    else:
        return "OK"

same logic, same result - a block of statements vs one expression
```

### Your turn
Write `classify` as a lambda with a chained conditional, and check it gives the same result as the `def` version for every row.

In [4]:
classify_lambda = lambda p: "OVER LIMIT" if p >= 100 else "WARNING" if p >= 90 else "OK"
servers["status_lambda"] = servers["percent"].apply(classify_lambda)
print((servers["status"] == servers["status_lambda"]).all())

True


---
## 3. `.apply()` with a lambda, and `sort_values(key=...)`

lambda earns its place exactly here — a function small enough to write inline,
used exactly once, right where it is needed.

In [5]:
servers["status"] = servers["percent"].apply(
    lambda p: "OVER LIMIT" if p >= 100 else "WARNING" if p >= 90 else "OK"
)
print(servers.sort_values("hostname", key=lambda col: col.str.lower()))

  hostname region  used  total  percent      status           used_level  \
0   srv-01   east    87    100     87.0          OK       below capacity   
1   srv-02   east    45    100     45.0          OK       below capacity   
2   srv-03   west   120    100    120.0  OVER LIMIT  at or over capacity   
3   srv-04   west    93    100     93.0     WARNING       below capacity   
4   srv-05  north   101    100    101.0  OVER LIMIT  at or over capacity   
5   srv-06   east    60    100     60.0          OK       below capacity   
6   srv-07   west    30    100     30.0          OK       below capacity   
7   srv-08  north   110    100    110.0  OVER LIMIT  at or over capacity   
8   srv-09   east    95    100     95.0     WARNING       below capacity   
9   srv-10  north    55    100     55.0          OK       below capacity   

  status_lambda  
0            OK  
1            OK  
2    OVER LIMIT  
3       WARNING  
4    OVER LIMIT  
5            OK  
6            OK  
7    OVER LIMIT  
8

### Your turn
Use `.apply()` with a lambda to add a column `region_upper` — the `region` column, in capitals.

In [6]:
servers["region_upper"] = servers["region"].apply(lambda region: region.upper())
print(servers[["region", "region_upper"]])

  region region_upper
0   east         EAST
1   east         EAST
2   west         WEST
3   west         WEST
4  north        NORTH
5   east         EAST
6   west         WEST
7  north        NORTH
8   east         EAST
9  north        NORTH


---
## 4. Summarising, saving, and a first chart

`describe()` for a quick overview, `.to_csv()` to save, and one chart — straight
off a DataFrame. In a notebook, calling `.plot()` shows the chart inline.

In [7]:
import matplotlib.pyplot as plt

print(servers.describe())

servers.to_csv("servers_report.csv", index=False)

servers["status"].value_counts().plot(kind="bar", title="Servers by status")
plt.tight_layout()
plt.show()

Matplotlib is building the font cache; this may take a moment.


            used  total    percent
count   10.00000   10.0   10.00000
mean    79.60000  100.0   79.60000
std     30.05994    0.0   30.05994
min     30.00000  100.0   30.00000
25%     56.25000  100.0   56.25000
50%     90.00000  100.0   90.00000
75%     99.50000  100.0   99.50000
max    120.00000  100.0  120.00000


C:\Users\patry\AppData\Local\Temp\ipykernel_32764\924684045.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Your turn
Make a bar chart of servers per region instead of per status.

In [8]:
servers["region"].value_counts().plot(kind="bar", title="Servers per region")
plt.tight_layout()
plt.show()

C:\Users\patry\AppData\Local\Temp\ipykernel_32764\2740410352.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Topic 4 Challenge

Load `servers.csv`, add `percent` and `status` (via `.apply()` with a lambda),
save the result to a new CSV, and produce one bar chart of the status counts.

In [9]:
import matplotlib.pyplot as plt
servers = pd.read_csv("servers.csv")
servers["percent"] = (servers["used"] / servers["total"]) * 100
servers["status"] = servers["percent"].apply(lambda p: "OVER LIMIT" if p >= 100 else "WARNING" if p >= 90 else "OK")
servers.to_csv("servers_report.csv", index=False)
servers["status"].value_counts().plot(kind="bar", title="Server status counts")
plt.tight_layout()
plt.savefig("server_status_counts.png")
plt.show()

C:\Users\patry\AppData\Local\Temp\ipykernel_32764\993883127.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Same idea, three fields

| Field | Looks like |
|---|---|
| **AI / Data Science** | `df["clean_text"] = df["text"].apply(lambda t: t.strip().lower())` |
| **Cyber Security** | `df["flag"] = df["failed"].apply(lambda n: n >= 3)` |
| **IT** | `df.sort_values("hostname", key=lambda c: c.str.lower())` |

---
## Common mistakes

| Mistake | What you see | Fix |
|---|---|---|
| a lambda with an `if` statement (not an expression) | `SyntaxError` | lambda only allows one expression — use chained `if/else`, or write a `def` |
| reusing the same lambda logic in three places | works, but repeats the logic three times | that is the sign you wanted a named function, not a lambda |
| forgetting `.apply()` and just writing `column.classify()` | `AttributeError` | `.apply(function)` runs your function on every value — the function itself is not a method on the column |
| calling `.plot()` in a plain script and seeing nothing | no chart appears | add `plt.show()` (script) — a notebook shows it automatically |

## Go deeper

`examples/01_apply_and_lambda.py` — apply, lambda syntax, chained conditionals, sort key
`examples/02_summarizing_saving_and_charting.py` — describe, to_csv, a first chart

---
## Recap

| You can now | Using |
|---|---|
| apply a function to a whole column | `column.apply(function)` |
| write a small inline function | `lambda x: expression` |
| choose lambda vs a named function | one-line, used once → lambda; reused or complex → `def` |
| get a quick overview | `.describe()` |
| save a result | `.to_csv(path, index=False)` |
| chart straight from a DataFrame | `.plot(kind=...)` |

**You now have all of CW1's toolkit.** Your mini-project this week closes it out.